# Flipkart Gridlock 2.0: V13 Asymmetric Graph Architecture
## Spatial Spillover & Temporal Trajectory (Momentum)

**Architectural Paradigm:**
This pipeline abandons the assumption of isolated geohashes. It mathematically links the city into a continuous spatial graph, allowing the models to detect traffic spillover from physically adjacent intersections. Furthermore, it shifts from static memory to dynamic trajectory by calculating the momentum (derivative) of demand. To penalize the under-prediction of severe traffic jams, the LightGBM engine is augmented with a custom Asymmetric Mean Squared Error objective function.

In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

# 1. System Initialization & Matrix Ingestion
data_paths = [".", "data/raw", "../../data/raw"]
base_path = next((path for path in data_paths if os.path.exists(os.path.join(path, "train.csv"))), None)

if base_path is None:
    raise FileNotFoundError("System could not locate the source data matrices.")

raw_train = pd.read_csv(os.path.join(base_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(base_path, "test.csv"))

y_train = raw_train['demand'].values
submission_index = raw_test['Index'].values

print("System initialized. Level-1 base matrices loaded.")

# 2. Spatial Graph & Momentum Engineering
def engineer_graph_features(source_df, target_df):
    """
    Injects 24H/48H lags, calculates demand momentum, and computes
    spatial spillover from the 8 physically adjacent geohash blocks.
    """
    df_f = target_df.copy()
    
    # 2a. Static Autoregressive Memory
    lag_df = source_df[['geohash', 'day', 'timestamp', 'demand']].copy()
    
    lag_24 = lag_df.copy()
    lag_24['day'] += 1
    lag_24.rename(columns={'demand': 'lag_24h'}, inplace=True)
    
    lag_48 = lag_df.copy()
    lag_48['day'] += 2
    lag_48.rename(columns={'demand': 'lag_48h'}, inplace=True)
    
    df_f = df_f.merge(lag_24, on=['geohash', 'day', 'timestamp'], how='left')
    df_f = df_f.merge(lag_48, on=['geohash', 'day', 'timestamp'], how='left')
    df_f[['lag_24h', 'lag_48h']] = df_f[['lag_24h', 'lag_48h']].fillna(0.0)
    
    # 2b. Temporal Momentum (The Derivative of Demand)
    # Measures the trajectory of traffic growth between T-48 and T-24
    df_f['momentum_24_48'] = df_f['lag_24h'] - df_f['lag_48h']
    
    # 2c. Spatial Spillover Mapping (Graph Adjacency)
    # We construct a hash map for O(1) lookups during the graph traversal
    lag_24_lookup = lag_24.set_index(['geohash', 'day', 'timestamp'])['lag_24h'].to_dict()
    
    def compute_spillover(row):
        try:
            # Retrieve the 8 physically touching geohashes
            neighbors = pgh.neighbors(row['geohash'])
            spillover_sum = 0.0
            valid_nodes = 0
            
            for neighbor_node in neighbors:
                temporal_key = (neighbor_node, row['day'], row['timestamp'])
                if temporal_key in lag_24_lookup:
                    spillover_sum += lag_24_lookup[temporal_key]
                    valid_nodes += 1
                    
            return spillover_sum / valid_nodes if valid_nodes > 0 else 0.0
        except Exception:
            return 0.0
            
    print("Computing Adjacency Spillover Graph...")
    df_f['neighbor_spillover_24h'] = df_f.apply(compute_spillover, axis=1)
    
    # 2d. Spatiotemporal Kinematics
    t_split = df_f['timestamp'].str.split(':', expand=True).astype(int)
    df_f['ts_minutes'] = t_split[0] * 60 + t_split[1]
    df_f['hour'] = t_split[0]
    df_f['time_slot_15m'] = df_f['ts_minutes'] // 15
    
    df_f['hour_sin'] = np.sin(2 * np.pi * df_f['hour'] / 24.0)
    df_f['hour_cos'] = np.cos(2 * np.pi * df_f['hour'] / 24.0)
    
    # 2e. Interaction Keys & Imputation
    df_f['geo_time_interaction'] = df_f['geohash'].astype(str) + "_" + df_f['time_slot_15m'].astype(str)
    df_f['is_rush_hour'] = df_f['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    
    df_f['Temperature'] = df_f['Temperature'].fillna(df_f['Temperature'].median())
    for col in ['Weather', 'RoadType', 'LargeVehicles', 'Landmarks']:
        if col in df_f.columns:
            df_f[col] = df_f[col].fillna('Unknown')
            
    return df_f

X_train_fe = engineer_graph_features(raw_train, raw_train.drop(columns=['demand'], errors='ignore'))
X_test_fe = engineer_graph_features(raw_train, raw_test.drop(columns=['Index'], errors='ignore'))

System initialized. Level-1 base matrices loaded.
Computing Adjacency Spillover Graph...
Computing Adjacency Spillover Graph...


### Phase 2: Asymmetric Engine Compilation & Alignment
To prevent the temporal drift observed in Level-2 stacking, the pipeline reverts to a robust Nelder-Mead optimization solver. Crucially, the LightGBM objective function is overridden with a custom asymmetric gradient. By applying a mathematical penalty multiplier to positive residuals (instances where actual traffic was higher than predicted), the engine is forced to prioritize the identification of anomalous traffic spikes over baseline accuracy.

In [4]:
# 3. Leak-Free Out-Of-Fold Target Encoding
def apply_oof_encoding(tr_df, te_df, tgt, col, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    tr_enc = np.zeros(len(tr_df))
    
    tmp_tr = tr_df[[col]].copy()
    tmp_tr['tgt'] = tgt
    g_mean = tgt.mean()
    
    for tr_idx, val_idx in kf.split(tmp_tr):
        f_map = tmp_tr.iloc[tr_idx].groupby(col)['tgt'].mean()
        tr_enc[val_idx] = tmp_tr.iloc[val_idx][col].map(f_map).fillna(g_mean).values
        
    te_map = tmp_tr.groupby(col)['tgt'].mean()
    te_enc = te_df[col].map(te_map).fillna(g_mean).values
        
    return tr_enc, te_enc

X_train_fe['TE_geo_time'], X_test_fe['TE_geo_time'] = apply_oof_encoding(X_train_fe, X_test_fe, y_train, 'geo_time_interaction')
X_train_fe['TE_geohash'], X_test_fe['TE_geohash'] = apply_oof_encoding(X_train_fe, X_test_fe, y_train, 'geohash')

cat_features = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_features:
    le = LabelEncoder()
    le.fit(X_train_fe[c].astype(str).tolist() + X_test_fe[c].astype(str).tolist())
    X_train_fe[c] = le.transform(X_train_fe[c].astype(str))
    X_test_fe[c] = le.transform(X_test_fe[c].astype(str))

drop_columns = ['timestamp', 'geo_time_interaction', 'Index']
features = [c for c in X_train_fe.columns if c not in drop_columns]

X = X_train_fe[features].values
X_test = X_test_fe[features].values
X_test = np.nan_to_num(X_test, nan=0.0)

# 4. Custom Asymmetric Objective Function (Prioritizes Traffic Jam Detection)
def asymmetric_mse(preds, train_data):
    y_true = train_data.get_label()
    residual = (y_true - preds).astype("float")
    # Apply a 1.5x gradient penalty if the model under-predicts the traffic
    grad = np.where(residual > 0, -2.0 * 1.5 * residual, -2.0 * residual)
    hess = np.where(residual > 0, 2.0 * 1.5, 2.0)
    return grad, hess

# 5. Algorithmic Triad Execution
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_params = {
    'objective': asymmetric_mse, 
    'metric': 'rmse',
    'learning_rate': 0.03, 
    'max_depth': 8, 
    'num_leaves': 128, 
    'min_child_samples': 20, 
    'verbose': -1, 
    'random_state': 42, 
    'n_jobs': -1
}

xgb_params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 7, 'random_state': 42, 'n_jobs': -1}
cat_params = {'iterations': 2500, 'learning_rate': 0.03, 'depth': 8, 'eval_metric': 'RMSE', 'verbose': 0, 'random_seed': 42}

oof_lgb, test_lgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_xgb, test_xgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_cat, test_cat = np.zeros(len(X)), np.zeros(len(X_test))

print("Initiating V13 Asymmetric Triad Training...")

for fold, (t_idx, v_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X[t_idx], y_train[t_idx]
    X_va, y_va = X[v_idx], y_train[v_idx]
    
    m_lgb = lgb.train(
        lgb_params, 
        lgb.Dataset(X_tr, y_tr), 
        num_boost_round=2500, 
        valid_sets=[lgb.Dataset(X_va, y_va)], 
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    oof_lgb[v_idx] = m_lgb.predict(X_va)
    test_lgb += m_lgb.predict(X_test) / 5
    
    m_xgb = xgb.train(xgb_params, xgb.DMatrix(X_tr, y_tr), 2500, evals=[(xgb.DMatrix(X_va, y_va), 'val')], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb[v_idx] = m_xgb.predict(xgb.DMatrix(X_va))
    test_xgb += m_xgb.predict(xgb.DMatrix(X_test)) / 5
    
    m_cat = CatBoostRegressor(**cat_params).fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_cat[v_idx] = m_cat.predict(X_va)
    test_cat += m_cat.predict(X_test) / 5
    
    print(f"Fold {fold+1} validation finalized.")

# 6. Nelder-Mead Convergence & Array Assembly
def optimization_objective(weights):
    weights = np.array(weights)
    if weights.sum() == 0: return 999.0
    normalized_weights = weights / weights.sum()
    blended_prediction = (normalized_weights[0] * oof_lgb) + (normalized_weights[1] * oof_xgb) + (normalized_weights[2] * oof_cat)
    return -max(0, 100 * r2_score(y_train, blended_prediction))

optimal_weights = minimize(optimization_objective, [0.33, 0.33, 0.33], method='Nelder-Mead').x
optimal_weights /= sum(optimal_weights)

final_test_predictions = np.clip((optimal_weights[0] * test_lgb) + (optimal_weights[1] * test_xgb) + (optimal_weights[2] * test_cat), 0.0, 1.0)
final_r2 = max(0, 100 * r2_score(y_train, (optimal_weights[0] * oof_lgb) + (optimal_weights[1] * oof_xgb) + (optimal_weights[2] * oof_cat)))

submission_payload = pd.DataFrame({'Index': submission_index, 'demand': final_test_predictions})
submission_payload.to_csv("submission_v13.csv", index=False)

print("\n==================================================")
print("PIPELINE EXECUTION COMPLETE (V13 GRAPH ARCHITECTURE)")
print("==================================================")
print(f"Optimal Engine Distribution: LGBM: {optimal_weights[0]:.3f} | XGB: {optimal_weights[1]:.3f} | CAT: {optimal_weights[2]:.3f}")
print(f"Terminal R2 Score: {final_r2:.4f}")
print("Output Matrix Generated: submission_v13.csv")

Initiating V13 Asymmetric Triad Training...
Fold 1 validation finalized.
Fold 2 validation finalized.
Fold 3 validation finalized.
Fold 4 validation finalized.
Fold 5 validation finalized.

PIPELINE EXECUTION COMPLETE (V13 GRAPH ARCHITECTURE)
Optimal Engine Distribution: LGBM: 0.093 | XGB: 0.608 | CAT: 0.298
Terminal R2 Score: 94.9787
Output Matrix Generated: submission_v13_graph_asymmetric.csv
